In [ ]:
%%capture
import pandas as pd
from pathlib import Path
import plotly.express as px

import torch
from nnsight import LanguageModel
from transformers import AutoTokenizer

dir = Path('../data/transcripts')
doc_paths = sorted(dir.glob('*.txt'))

# tokenize + collect activations
# id = 'allenai/Olmo-3-1125-32B'
id = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
model = LanguageModel(id, device_map = 'auto', remote = 'true')
tokenizer = AutoTokenizer.from_pretrained(id)

In [ ]:
with open(doc_paths[3], "r", encoding="utf-8") as f:
    michael = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

with open(doc_paths[4], "r", encoding="utf-8") as f:
    mike = tokenizer.apply_chat_template([{'role': 'user', 'content': 'Think about this conversation and try your best to distinguish the people who are involved' + f.read()}], tokenize = False)

In [ ]:
# get tokens
michael_toks = tokenizer(michael, return_offsets_mapping = True)
mike_toks = tokenizer(mike, return_offsets_mapping = True)

def get_token_map(token_dict, string):
    token_map = {}
    token_map['token_id'] = token_dict['input_ids']
    token_map['offsets'] = token_dict['offset_mapping']
    token_map['token'] = [string[o[0]:o[1]] for o in token_map['offsets']]
    token_map['token_ix'] = [i for i in range(len(token_map['token']))]
    return pd.DataFrame(token_map)

michael_map = get_token_map(michael_toks, michael)
mike_map = get_token_map(mike_toks, mike)


In [ ]:
# get hidden states
michael_hs = None
with model.trace(michael, remote = True):
    michael_hs = model.model.layers[24].output[0].save()

mike_hs = None
with model.trace(mike, remote = True):
    mike_hs = model.model.layers[24].output[0].save()


In [ ]:
# now, we want a way to indicate which segments are "where the experiment is happening"
from utils.utils import flag_message_types
michael_map_labeled = flag_message_types(michael_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])
mike_map_labeled = flag_message_types(mike_map, base_messages = ['oh, well have fun at tennis, I think later we will go to the Fresh Market and get some pasta and salmon, goodbye!'])


In [ ]:
import numpy as np
import torch
from sklearn.decomposition import PCA

skip_n = 11

# Slice token maps to post-skip_n portion (aligns with hs tensor indices)
michael_map_slice = michael_map_labeled.iloc[skip_n:].reset_index(drop=True)
mike_map_slice = mike_map_labeled.iloc[skip_n:].reset_index(drop=True)

# Filter to experimental segment only
michael_exp_mask = michael_map_slice['base_message_ix'] == 0.0
mike_exp_mask = mike_map_slice['base_message_ix'] == 0.0

michael_exp_pos = michael_exp_mask[michael_exp_mask].index.tolist()
mike_exp_pos = mike_exp_mask[mike_exp_mask].index.tolist()

# Extract hidden states only for experimental segment tokens
michael_exp_hs = michael_hs[skip_n:][michael_exp_pos].float().numpy()
mike_exp_hs = mike_hs[skip_n:][mike_exp_pos].float().numpy()

# Fit PCA only on these points
comb = np.concatenate([michael_exp_hs, mike_exp_hs], axis=0)
pca = PCA(n_components=2)
coords = pca.fit_transform(comb)

michael_hs_pca = pd.DataFrame(coords[:len(michael_exp_pos), :])
mike_hs_pca = pd.DataFrame(coords[len(michael_exp_pos):, :])

print(michael_hs_pca, mike_hs_pca)

In [ ]:
# plotting — only experimental segment tokens
michael_plot = pd.concat([michael_map_slice[michael_exp_mask].reset_index(drop=True), michael_hs_pca], axis=1)
michael_plot['is_sequence'] = 'Michael'

mike_plot = pd.concat([mike_map_slice[mike_exp_mask].reset_index(drop=True), mike_hs_pca], axis=1)
mike_plot['is_sequence'] = 'Mike'

In [ ]:
plot_df = pd.concat([michael_plot, mike_plot], axis=0)
plot_df['color'] = plot_df['is_sequence']

In [ ]:
import plotly.express as px
plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
plot

In [ ]:
new_plot = px.scatter(plot_df, x = plot_df[0], y = plot_df[1], color = 'color', hover_data = ['token'])
new_plot

In [ ]:
# this doesn't show much, but we expect the variance to be pretty small because it'll emphasize the 
# directions of highest variance (which could result from something else :))
plot